In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
from unidecode import unidecode

### ORDERS

In [ ]:
data_orders = pd.read_csv("../db/olist_orders_dataset.csv")
data_orders.shape

(99441, 8)

In [3]:
data_orders = data_orders.drop(columns = ["order_approved_at","order_delivered_carrier_date"])
data_orders.shape

(99441, 6)

In [4]:
data_orders.duplicated().sum()

np.int64(0)

In [5]:
data_orders["order_purchase_timestamp"] = pd.to_datetime(data_orders["order_purchase_timestamp"]).dt.normalize() # datetime64[ns]
data_orders["order_delivered_customer_date"] = pd.to_datetime(data_orders["order_delivered_customer_date"]).dt.normalize()
data_orders["order_estimated_delivery_date"] = pd.to_datetime(data_orders["order_estimated_delivery_date"]).dt.normalize()
data_orders["order_status"] = data_orders["order_status"].apply(unidecode).str.strip().str.lower()
data_orders["delivered_status"] = np.where(data_orders["order_status"] == "delivered",1,0)
data_orders["delay_time"] = (data_orders["order_estimated_delivery_date"] - data_orders["order_delivered_customer_date"]).dt.days.astype("Int64")
data_orders = data_orders.drop(columns = ["order_status", "order_estimated_delivery_date", "order_delivered_customer_date"])

In [6]:
data_orders.loc[data_orders["delivered_status"] == 1]

,order_id,customer_id,order_purchase_timestamp,delivered_status,delay_time
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02,1,8
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24,1,6
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08,1,18
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18,1,13
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13,1,10
...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,2017-03-09,1,11
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,2018-02-06,1,2
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,2017-08-27,1,6
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,2018-01-08,1,21


In [7]:
data_orders.head()

,order_id,customer_id,order_purchase_timestamp,delivered_status,delay_time
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02,1,8
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24,1,6
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08,1,18
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18,1,13
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13,1,10


In [8]:
data_orders.nunique()

order_id                    99441
customer_id                 99441
order_purchase_timestamp      634
delivered_status                2
delay_time                    198
dtype: int64

In [9]:
data_orders.isna().sum()

order_id                       0
customer_id                    0
order_purchase_timestamp       0
delivered_status               0
delay_time                  2965
dtype: int64

In [10]:
data_orders["delay_time"] = data_orders["delay_time"].fillna(-1000)
data_orders.isna().sum()

order_id                    0
customer_id                 0
order_purchase_timestamp    0
delivered_status            0
delay_time                  0
dtype: int64

In [11]:
# dropear valores que no tengamos customer date
#time_mode_purchase_to_delivered = data_orders["order_delivered_customer_date"] - data_orders["order_purchase_timestamp"]
#mode = time_mode_purchase_to_delivered.mode()
#mode

In [12]:
#data_orders["order_delivered_customer_date"] = data_orders["order_delivered_customer_date"].fillna(data_orders["order_purchase_timestamp"] + mode[0])
data_orders.isna().sum()

order_id                    0
customer_id                 0
order_purchase_timestamp    0
delivered_status            0
delay_time                  0
dtype: int64

### PRODUCTS + CATEGORY

In [ ]:
data_products = pd.read_csv("../db/olist_products_dataset.csv")
data_products = data_products.drop(columns = "product_name_lenght")
data_products.dtypes

product_id                     object
product_category_name          object
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object

In [14]:
data_products["product_weight_g"].mode()

0    200.0
Name: product_weight_g, dtype: float64

In [15]:
measure_list = ["product_description_lenght", "product_photos_qty", "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
mode_measures = {col: data_products[col].mode()[0] for col in measure_list}
print(mode_measures)

{'product_description_lenght': np.float64(404.0), 'product_photos_qty': np.float64(1.0), 'product_weight_g': np.float64(200.0), 'product_length_cm': np.float64(16.0), 'product_height_cm': np.float64(10.0), 'product_width_cm': np.float64(11.0)}


In [16]:
data_products.isna().sum()

product_id                      0
product_category_name         610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [17]:
data_products["product_description_lenght"].unique()

array([ 287.,  276.,  250., ..., 2836., 3364., 2207.])

In [18]:
data_products.dtypes

product_id                     object
product_category_name          object
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object

In [ ]:
data_category = pd.read_csv("../db/product_category_name_translation.csv")

nuevas_filas = pd.DataFrame(
    {"product_category_name" : ["pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos"],
     "product_category_name_english" : ["pc_gamer", "portable_kitchen_and_food_preparators"]
    }
)
data_category = pd.concat([data_category, nuevas_filas])
products_merged = data_products.merge(
    data_category,
    on = "product_category_name",
    how = "left"
)
products_merged.drop(columns = "product_category_name", inplace = True)
data_products = products_merged
data_products["product_category_name_english"] = data_products["product_category_name_english"].fillna("unknown")
data_products

,product_id,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,402.0,4.0,625.0,20.0,17.0,13.0,housewares
...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,67.0,2.0,12300.0,40.0,40.0,40.0,furniture_decor
32947,bf4538d88321d0fd4412a93c974510e6,971.0,1.0,1700.0,16.0,19.0,16.0,construction_tools_lights
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,799.0,1.0,1400.0,27.0,7.0,27.0,bed_bath_table
32949,83808703fc0706a22e264b9d75f04a2e,156.0,2.0,700.0,31.0,13.0,20.0,computers_accessories


In [20]:
data_products["product_category_name_english"].unique()

array(['perfumery', 'art', 'sports_leisure', 'baby', 'housewares',
       'musical_instruments', 'cool_stuff', 'furniture_decor',
       'home_appliances', 'toys', 'bed_bath_table',
       'construction_tools_safety', 'computers_accessories',
       'health_beauty', 'luggage_accessories', 'garden_tools',
       'office_furniture', 'auto', 'electronics', 'fashion_shoes',
       'telephony', 'stationery', 'fashion_bags_accessories', 'computers',
       'home_construction', 'watches_gifts',
       'construction_tools_construction', 'pet_shop', 'small_appliances',
       'agro_industry_and_commerce', 'unknown', 'furniture_living_room',
       'signaling_and_security', 'air_conditioning', 'consoles_games',
       'books_general_interest', 'costruction_tools_tools',
       'fashion_underwear_beach', 'fashion_male_clothing',
       'kitchen_dining_laundry_garden_furniture',
       'industry_commerce_and_business', 'fixed_telephony',
       'construction_tools_lights', 'books_technical',
     

In [21]:
data_products.isna().sum()

product_id                         0
product_description_lenght       610
product_photos_qty               610
product_weight_g                   2
product_length_cm                  2
product_height_cm                  2
product_width_cm                   2
product_category_name_english      0
dtype: int64

### CUSTOMERS

In [ ]:
data_customers = pd.read_csv("../db/olist_customers_dataset.csv")
data_customers.shape

(99441, 5)

In [23]:
data_customers = data_customers.drop(columns = "customer_city")
data_customers["customer_state"] = data_customers["customer_state"].apply(unidecode).str.strip()
data_customers = data_customers.drop(columns ="customer_zip_code_prefix")

In [24]:
data_customers.columns

Index(['customer_id', 'customer_unique_id', 'customer_state'], dtype='object')

### REVIEW

In [ ]:
data_reviews = pd.read_csv("../db/olist_order_reviews_dataset.csv")

data_reviews = data_reviews.drop(columns =["review_comment_title", "review_comment_message"])
data_reviews["review_creation_date"] = pd.to_datetime(data_reviews["review_creation_date"], format="%Y-%m-%d %H:%M:%S").dt.normalize()
data_reviews["review_answer_timestamp"] = pd.to_datetime(data_reviews["review_answer_timestamp"], format="%Y-%m-%d %H:%M:%S").dt.normalize()

data_reviews = (
    data_reviews
    .groupby("order_id")
    .agg(
        media_review_score=("review_score", "mean"),
        fecha_ultima_review=("review_creation_date", "max") 
    )
    .reset_index()
)

data_reviews["media_review_score"] = pd.to_numeric(data_reviews["media_review_score"], errors="coerce").astype("Float64").astype("Int64")

data_reviews.shape

(98673, 3)

In [26]:
data_reviews.isna().sum()

order_id               0
media_review_score     0
fecha_ultima_review    0
dtype: int64

In [27]:
data_reviews.head()

,order_id,media_review_score,fecha_ultima_review
0,00010242fe8c5a6d1ba2dd792cb16214,5,2017-09-21
1,00018f77f2f0320c557190d7a144bdd3,4,2017-05-13
2,000229ec398224ef6ca0657da4fc703e,5,2018-01-23
3,00024acbcdf0a6daa1e931b038114c75,4,2018-08-15
4,00042b26cf59d7ce69dfabb4e55b4fd9,5,2017-03-02


### PAYMENTS

In [ ]:
data_pay = pd.read_csv("../db/olist_order_payments_dataset.csv")

data_pay_grouped = data_pay.groupby(by = "order_id").agg(
    number_payments = ("order_id", "size"),
    payment_value_sum = ("payment_value", "sum"),
    number_payment_types = ("payment_type", "nunique"),
    payment_type_min = ("payment_type", "min")).reset_index()

data_pay_grouped = data_pay_grouped.assign(
    payment_type = np.where(
        data_pay_grouped["number_payment_types"] > 1,
        "multiple_payments",
        data_pay_grouped["payment_type_min"]
    )
)

data_pay = data_pay_grouped.drop(columns = ["payment_type_min", "number_payment_types"])
data_pay.shape

(99440, 4)

In [29]:
data_pay.head()

,order_id,number_payments,payment_value_sum,payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,1,72.19,credit_card
1,00018f77f2f0320c557190d7a144bdd3,1,259.83,credit_card
2,000229ec398224ef6ca0657da4fc703e,1,216.87,credit_card
3,00024acbcdf0a6daa1e931b038114c75,1,25.78,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,218.04,credit_card


### ORDER - CUSTOMERS - PAY

In [30]:
data_orders.shape

(99441, 5)

In [31]:
data_orders = data_orders.merge(data_customers, on="customer_id", how = "left")
data_orders = data_orders.drop(columns = "customer_id")
data_orders.shape

(99441, 6)

In [32]:
data_orders = data_orders.merge(data_pay, on="order_id", how = "left")
data_orders.shape

(99441, 9)

In [33]:
data_orders.isna().sum()

order_id                    0
order_purchase_timestamp    0
delivered_status            0
delay_time                  0
customer_unique_id          0
customer_state              0
number_payments             1
payment_value_sum           1
payment_type                1
dtype: int64

### SELLERS

In [ ]:
data_sellers = pd.read_csv("../db/olist_sellers_dataset.csv")
data_sellers = data_sellers.drop(columns = ["seller_city","seller_zip_code_prefix"])
data_sellers.shape

(3095, 2)

In [35]:
data_sellers.columns

Index(['seller_id', 'seller_state'], dtype='object')

### ITEMS + PRODUCTS

In [ ]:
data_items = pd.read_csv("../db/olist_order_items_dataset.csv")
data_items = data_items.merge(data_products, on = "product_id", how = "left")
data_items.columns

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'product_category_name_english'],
      dtype='object')

In [37]:
top_category = data_items.loc[
    data_items.groupby("order_id")["price"].idxmax(),
    ["order_id", "product_category_name_english"]
]

data_items_grouped = (
    data_items
    .groupby("order_id")
    .agg(
        number_items=("product_id", "size"),
        total_price=("price", "sum"),
        total_freight_value=("freight_value", "sum"),
        total_diff_items=("product_id", "nunique"),
        product_id=("product_id", "first"),
        seller_id = ("seller_id", "first"),
        product_description_lenght =("product_description_lenght","max"),
        product_photos_qty =("product_photos_qty","max"),
        product_weight_g  =("product_weight_g","max"),
        product_length_cm =("product_length_cm","max"),
        product_height_cm =("product_height_cm","max"),
        product_width_cm =("product_width_cm","max")
    )
    .reset_index()
)

data_items_grouped = data_items_grouped.merge(
    top_category, on="order_id", how="left"
)

mask = data_items_grouped["total_diff_items"] > 1

data_items_grouped.loc[mask, "product_id"] = "multiple"
data_items_grouped.loc[mask, "seller_id"] = "multiple"

data_items = data_items_grouped
data_items.isna().sum()

order_id                            0
number_items                        0
total_price                         0
total_freight_value                 0
total_diff_items                    0
product_id                          0
seller_id                           0
product_description_lenght       1389
product_photos_qty               1389
product_weight_g                   16
product_length_cm                  16
product_height_cm                  16
product_width_cm                   16
product_category_name_english       0
dtype: int64

# Data FINAL

### Merge

In [38]:
final_data = data_reviews.merge(data_orders, on = "order_id", how = "left")
final_data.isna().sum()

order_id                    0
media_review_score          0
fecha_ultima_review         0
order_purchase_timestamp    0
delivered_status            0
delay_time                  0
customer_unique_id          0
customer_state              0
number_payments             1
payment_value_sum           1
payment_type                1
dtype: int64

In [39]:
final_data = final_data.merge(data_items, on = "order_id", how = "left")
final_data.isna().sum()

order_id                            0
media_review_score                  0
fecha_ultima_review                 0
order_purchase_timestamp            0
delivered_status                    0
delay_time                          0
customer_unique_id                  0
customer_state                      0
number_payments                     1
payment_value_sum                   1
payment_type                        1
number_items                      756
total_price                       756
total_freight_value               756
total_diff_items                  756
product_id                        756
seller_id                         756
product_description_lenght       2135
product_photos_qty               2135
product_weight_g                  772
product_length_cm                 772
product_height_cm                 772
product_width_cm                  772
product_category_name_english     756
dtype: int64

In [40]:
final_data = final_data.merge(data_sellers, on = "seller_id", how = "left")
final_data.isna().sum()

order_id                            0
media_review_score                  0
fecha_ultima_review                 0
order_purchase_timestamp            0
delivered_status                    0
delay_time                          0
customer_unique_id                  0
customer_state                      0
number_payments                     1
payment_value_sum                   1
payment_type                        1
number_items                      756
total_price                       756
total_freight_value               756
total_diff_items                  756
product_id                        756
seller_id                         756
product_description_lenght       2135
product_photos_qty               2135
product_weight_g                  772
product_length_cm                 772
product_height_cm                 772
product_width_cm                  772
product_category_name_english     756
seller_state                     3953
dtype: int64

### Clean

In [41]:
final_data.isna().sum()

order_id                            0
media_review_score                  0
fecha_ultima_review                 0
order_purchase_timestamp            0
delivered_status                    0
delay_time                          0
customer_unique_id                  0
customer_state                      0
number_payments                     1
payment_value_sum                   1
payment_type                        1
number_items                      756
total_price                       756
total_freight_value               756
total_diff_items                  756
product_id                        756
seller_id                         756
product_description_lenght       2135
product_photos_qty               2135
product_weight_g                  772
product_length_cm                 772
product_height_cm                 772
product_width_cm                  772
product_category_name_english     756
seller_state                     3953
dtype: int64

In [42]:
for col in measure_list:
    final_data[col] = final_data[col].fillna(mode_measures[col])

final_data["product_category_name_english"] = final_data["product_category_name_english"].fillna("unknown")
final_data["seller_state"] = final_data["seller_state"].fillna("unknown")
final_data["product_id"] = final_data["product_id"].fillna("unknown")
final_data["seller_id"] = final_data["seller_id"].fillna("unknown")

final_data.isna().sum()

order_id                           0
media_review_score                 0
fecha_ultima_review                0
order_purchase_timestamp           0
delivered_status                   0
delay_time                         0
customer_unique_id                 0
customer_state                     0
number_payments                    1
payment_value_sum                  1
payment_type                       1
number_items                     756
total_price                      756
total_freight_value              756
total_diff_items                 756
product_id                         0
seller_id                          0
product_description_lenght         0
product_photos_qty                 0
product_weight_g                   0
product_length_cm                  0
product_height_cm                  0
product_width_cm                   0
product_category_name_english      0
seller_state                       0
dtype: int64

In [43]:
final_data[["product_id","number_items","total_price","total_freight_value","total_diff_items"]].loc[final_data["number_items"].isna() & final_data["total_price"].isna() & final_data["total_freight_value"].isna() & final_data["total_diff_items"].isna()]

,product_id,number_items,total_price,total_freight_value,total_diff_items
24,unknown,NaN,NaN,NaN,NaN
227,unknown,NaN,NaN,NaN,NaN
247,unknown,NaN,NaN,NaN,NaN
267,unknown,NaN,NaN,NaN,NaN
305,unknown,NaN,NaN,NaN,NaN
...,...,...,...,...,...
97852,unknown,NaN,NaN,NaN,NaN
97879,unknown,NaN,NaN,NaN,NaN
98139,unknown,NaN,NaN,NaN,NaN
98196,unknown,NaN,NaN,NaN,NaN


In [44]:
final_data = final_data.dropna()
final_data.isna().sum()

order_id                         0
media_review_score               0
fecha_ultima_review              0
order_purchase_timestamp         0
delivered_status                 0
delay_time                       0
customer_unique_id               0
customer_state                   0
number_payments                  0
payment_value_sum                0
payment_type                     0
number_items                     0
total_price                      0
total_freight_value              0
total_diff_items                 0
product_id                       0
seller_id                        0
product_description_lenght       0
product_photos_qty               0
product_weight_g                 0
product_length_cm                0
product_height_cm                0
product_width_cm                 0
product_category_name_english    0
seller_state                     0
dtype: int64

In [45]:
final_data.shape

(97916, 25)

In [46]:
final_data.duplicated()

0        False
1        False
2        False
3        False
4        False
         ...  
98668    False
98669    False
98670    False
98671    False
98672    False
Length: 97916, dtype: bool

In [47]:
final_data.loc[final_data.duplicated(keep = False),:]

,order_id,media_review_score,fecha_ultima_review,order_purchase_timestamp,delivered_status,delay_time,customer_unique_id,customer_state,number_payments,payment_value_sum,...,product_id,seller_id,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_state


In [48]:
final_data.loc[final_data["order_id"] == "03aba68b07658f28f29612641f08d4ba"]

,order_id,media_review_score,fecha_ultima_review,order_purchase_timestamp,delivered_status,delay_time,customer_unique_id,customer_state,number_payments,payment_value_sum,...,product_id,seller_id,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_state
1412,03aba68b07658f28f29612641f08d4ba,4,2018-08-22,2018-08-08,1,6,c8460e4251689ba205045f3ea17884a1,RS,1.0,1202.64,...,e7cc48a9daff5436f63d3aad9426f28b,53243585a1d6dc2643021fd1853d8905,1500.0,4.0,700.0,32.0,15.0,21.0,telephony,BA


In [49]:
# Hay casos que no tienen order_id entre los items, pero si tienen valores evaluables, intentar extrapolar o analizar que hacer con esos nulos
# evaluar si tiene que ver con el delay time

In [50]:
final_data['seller_state'].head()

0    SP
1    SP
2    MG
3    SP
4    PR
Name: seller_state, dtype: object

### Reduccion categorias y regiones

In [51]:
mapeo_categorias = {
    'SP': 'Southeast', 
    'MG': 'Southeast', 
    'ES': 'Southeast', 
    'RS': 'South', 
    'DF': 'Central-West',
    'PR': 'South', 
    'SC': 'South', 
    'RJ': 'Southeast', 
    'GO': 'Central-West', 
    'BA': 'Northeast',
    'MA': 'Northeast', 
    'AC': 'North', 
    'PB': 'Northeast', 
    'PE': 'Northeast', 
    'CE': 'Northeast',
    'MT': 'Central-West', 
    'PI': 'Northeast', 
    'RN': 'Northeast', 
    'MS': 'Central-West', 
    'PA': 'North', 
    'AM': 'North',
    'SE': 'Northeast', 
    'RO': 'North',
    'RR': 'North',
    'TO': 'North',
    'AL': 'Northeast',
    'AP': 'North'
}

In [52]:
final_data['customer_state'] = final_data['customer_state'].astype(str)
final_data['seller_state']   = final_data['seller_state'].astype(str)
final_data['customer_region'] = final_data['customer_state'].replace(mapeo_categorias)
final_data['seller_region']   = final_data['seller_state'].replace(mapeo_categorias)
final_data[['customer_state', 'customer_region', 'seller_state', 'seller_region']].head(10)


,customer_state,customer_region,seller_state,seller_region
0,RJ,Southeast,SP,Southeast
1,SP,Southeast,SP,Southeast
2,MG,Southeast,MG,Southeast
3,SP,Southeast,SP,Southeast
4,SP,Southeast,PR,South
5,MG,Southeast,SP,Southeast
6,SP,Southeast,SP,Southeast
7,SP,Southeast,SP,Southeast
8,SP,Southeast,SP,Southeast
9,SP,Southeast,SP,Southeast


In [53]:
final_data["product_category_name_english"].nunique()

74

In [54]:
mapeo_categorias = {
    # 1. Home and Decoration
    'furniture_decor': 'Home and Decoration', 'housewares': 'Home and Decoration', 
    'bed_bath_table': 'Home and Decoration', 'office_furniture': 'Home and Decoration',
    'home_appliances': 'Home and Decoration', 'kitchen_dining_laundry_garden_furniture': 'Home and Decoration',
    'home_confort': 'Home and Decoration', 'fixed_telephony': 'Home and Decoration',
    'small_appliances_home_oven_and_coffee': 'Home and Decoration', 'home_construction': 'Home and Decoration',
    'furniture_living_room': 'Home and Decoration', 'home_appliances_2': 'Home and Decoration',
    'furniture_bedroom': 'Home and Decoration', 'home_comfort_2': 'Home and Decoration',
    'furniture_mattress_and_upholstery': 'Home and Decoration', 'la_cuisine': 'Home and Decoration',

    # 2. Electronics and Technology
    'telephony': 'Electronics and Technology', 'electronics': 'Electronics and Technology',
    'computers_accessories': 'Electronics and Technology', 'audio': 'Electronics and Technology',
    'watches_gifts': 'Electronics and Technology', 'computers': 'Electronics and Technology',
    'portable_kitchen_and_food_preparators': 'Electronics and Technology', 'dvds_blu_ray': 'Electronics and Technology',
    'cds_dvds_musicals': 'Electronics and Technology', 'tablets_printing_image': 'Electronics and Technology',
    'pc_gamer': 'Electronics and Technology',

    # 3. Fashion and Personal Care
    'perfumery': 'Fashion and Personal Care', 'health_beauty': 'Fashion and Personal Care',
    'fashion_bags_accessories': 'Fashion and Personal Care', 'luggage_accessories': 'Fashion and Personal Care',
    'fashion_underwear_beach': 'Fashion and Personal Care', 'fashion_male_clothing': 'Fashion and Personal Care',
    'fashion_shoes': 'Fashion and Personal Care', 'fashio_female_clothing': 'Fashion and Personal Care',
    'fashion_sport': 'Fashion and Personal Care', 'fashion_childrens_clothes': 'Fashion and Personal Care',
    'diapers_and_hygiene': 'Fashion and Personal Care', 'baby': 'Fashion and Personal Care',

    # 4. Leisure, Toys and Arts
    'sports_leisure': 'Leisure, Toys and Arts', 'consoles_games': 'Leisure, Toys and Arts',
    'toys': 'Leisure, Toys and Arts', 'musical_instruments': 'Leisure, Toys and Arts',
    'art': 'Leisure, Toys and Arts', 'books_general_interest': 'Leisure, Toys and Arts',
    'books_technical': 'Leisure, Toys and Arts', 'books_imported': 'Leisure, Toys and Arts',
    'party_supplies': 'Leisure, Toys and Arts', 'cine_photo': 'Leisure, Toys and Arts',
    'music': 'Leisure, Toys and Arts', 'arts_and_craftmanship': 'Leisure, Toys and Arts',

    # 5. Tools and Construction
    'garden_tools': 'Tools and Construction', 'construction_tools_construction': 'Tools and Construction',
    'construction_tools_lights': 'Tools and Construction', 'industry_commerce_and_business': 'Tools and Construction',
    'air_conditioning': 'Tools and Construction', 'signaling_and_security': 'Tools and Construction',
    'small_appliances': 'Tools and Construction', 'costruction_tools_garden': 'Tools and Construction',
    'construction_tools_safety': 'Tools and Construction', 'costruction_tools_tools': 'Tools and Construction',
    'security_and_services': 'Tools and Construction', 'agro_industry_and_commerce': 'Tools and Construction',

    # 6. Miscellaneous and Other Items
    'cool_stuff': 'Miscellaneous and Other Items', 'pet_shop': 'Miscellaneous and Other Items',
    'food': 'Miscellaneous and Other Items', 'food_drink': 'Miscellaneous and Other Items',
    'drinks': 'Miscellaneous and Other Items', 'stationery': 'Miscellaneous and Other Items',
    'auto': 'Miscellaneous and Other Items', 'unknown': 'Miscellaneous and Other Items',
    'market_place': 'Miscellaneous and Other Items', 'flowers': 'Miscellaneous and Other Items',
    'christmas_supplies': 'Miscellaneous and Other Items'
}

final_data["product_category_name_english"] = final_data["product_category_name_english"].replace(mapeo_categorias)
final_data["product_category_name_english"].nunique()

6

In [ ]:
final_data.to_csv("../db/output/final_data_dashboards.csv", index = False)